# 🎓 Student Performance Analytics & Exploratory Data Analysis (EDA)

## 📌 Objective
This notebook conducts an in-depth exploratory data analysis and statistical investigation into student academic and attendance data. Rather than merely viewing raw marks, we answer core institutional questions:
1. **Which students are performing well vs requiring immediate academic attention?**
2. **Which subjects have the lowest average marks and highest failure rates?**
3. **What is the statistical association between attendance and final exam performance?**
4. **How does performance vary across departments and sections?**
5. **How do students progress across assessment milestones (Mid 1 -> Mid 2 -> Final)?**

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Add src/ to path
sys.path.append(os.path.abspath("../src"))
from data_cleaning import DataCleaner
from data_processing import DataProcessor
from database import DatabaseManager

sns.set_theme(style="whitegrid", palette="muted")
print("Environment initialized successfully.")

## 1. 📥 Raw Data Ingestion & Data Hygiene Audit

In [ ]:
# Load raw datasets
df_raw_students = pd.read_csv("../data/raw/students.csv")
df_raw_marks = pd.read_csv("../data/raw/marks.csv")
df_raw_attendance = pd.read_csv("../data/raw/attendance.csv")

print(f"Raw Students Shape: {df_raw_students.shape}")
print(f"Raw Marks Shape: {df_raw_marks.shape}")
print(f"Raw Attendance Shape: {df_raw_attendance.shape}")

print("\n--- Missing Values Audit ---")
print("Students Missing:\n", df_raw_students.isnull().sum())
print("Marks Missing:\n", df_raw_marks.isnull().sum())
print("Attendance Missing:\n", df_raw_attendance.isnull().sum())

## 2. 🧹 Data Cleaning & Anomaly Correction
We use our automated `DataCleaner` module to deduplicate, impute missing values, and clip out-of-range marks/attendance.

In [ ]:
cleaner = DataCleaner(raw_data_dir="../data/raw")
df_students_clean, df_marks_clean, df_att_clean, audit_log = cleaner.clean_all(save_processed=False)
pd.DataFrame(audit_log)

## 3. ⚙️ Feature Engineering & Metric Calculation

In [ ]:
processor = DataProcessor()
df_details, df_summary = processor.process_and_merge(df_students_clean, df_marks_clean, df_att_clean, save_master=False)
df_summary.head()

## 4. 📊 Core Statistical Insights & Distributions

In [ ]:
print(f"Overall Average Score: {df_summary['average_marks'].mean():.2f}%")
print(f"Median Score: {df_summary['average_marks'].median():.2f}%")
print(f"Score Standard Deviation: {df_summary['average_marks'].std():.2f}")
print(f"Pass Rate: {((df_summary['pass_status'] == 'Passed').mean() * 100):.1f}%")
print(f"Average Attendance: {df_summary['overall_attendance_pct'].mean():.1f}%")

# Distribution plot
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df_summary["average_marks"], kde=True, ax=ax[0], color="#2563EB", bins=15)
ax[0].set_title("Distribution of Student Average Marks (%)")
ax[0].axvline(df_summary["average_marks"].mean(), color="red", linestyle="--", label=f"Mean: {df_summary['average_marks'].mean():.1f}%")
ax[0].legend()

sns.histplot(df_summary["overall_attendance_pct"], kde=True, ax=ax[1], color="#10B981", bins=15)
ax[1].set_title("Distribution of Student Attendance (%)")
ax[1].axvline(75, color="red", linestyle="--", label="75% Mandatory Cutoff")
ax[1].legend()
plt.tight_layout()
plt.show()

## 5. ⏱️ Attendance vs Academic Performance Correlation
> **Methodological Note:** We investigate the statistical correlation while recognizing that attendance serves as an indicator of academic engagement rather than direct mechanical causation.

In [ ]:
r, p_val = stats.pearsonr(df_summary["overall_attendance_pct"], df_summary["average_marks"])
print(f"Pearson correlation coefficient: r = {r:.3f}, p-value = {p_val:.4e}")

plt.figure(figsize=(9, 6))
sns.regplot(
    data=df_summary, 
    x="overall_attendance_pct", 
    y="average_marks", 
    scatter_kws={"alpha": 0.6, "color": "#2563EB"}, 
    line_kws={"color": "#EF4444", "linewidth": 2}
)
plt.axvline(75, color="grey", linestyle=":", label="75% Attendance Threshold")
plt.title(f"Attendance vs Academic Performance (r = {r:.3f})", fontsize=14)
plt.xlabel("Overall Attendance (%)")
plt.ylabel("Average Academic Score (%)")
plt.legend()
plt.show()

## 6. 📚 Subject-wise Difficulty & Failure Rates

In [ ]:
subj_analysis = df_details.groupby("subject").agg(
    avg_score=("normalized_score", "mean"),
    pass_rate=("subject_passed", lambda x: (x.mean() * 100)),
    avg_final=("final", "mean")
).reset_index().sort_values(by="avg_score", ascending=True)

plt.figure(figsize=(10, 5))
sns.barplot(data=subj_analysis, x="avg_score", y="subject", palette="Blues_r")
plt.title("Subject Difficulty Ranking (Ascending Average Score)")
plt.xlabel("Average Normalized Score (%)")
plt.xlim(0, 100)
for index, row in subj_analysis.iterrows():
    plt.text(row["avg_score"] + 1, index, f"{row['avg_score']:.1f}% (Pass: {row['pass_rate']:.1f}%)", va="center")
plt.show()

## 7. 🏛️ Departmental Benchmarking

In [ ]:
dept_perf = df_summary.groupby("department").agg(
    students=("student_id", "count"),
    avg_marks=("average_marks", "mean"),
    avg_attendance=("overall_attendance_pct", "mean"),
    pass_rate=("all_passed", lambda x: (x.mean() * 100))
).round(2)
dept_perf